# Run Backtest Rank

Run z-score mean-reversion backtests on all cointegrated pairs (or a capped subset), then show the **top 10 pairs by Sharpe ratio**. Use the printed `PAIRS` list in `pairs_backtest.ipynb`.

In [13]:
import sys
from datetime import timedelta
from pathlib import Path

import pandas as pd

_root = Path.cwd().resolve()
while _root != _root.parent and not (_root / ".git").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

from research.functions.load_data import load_prices
from research.functions.backtest_pair import backtest_pair

PROCESSED_DIR = _root / "research" / "processed"
DATA_DIR = _root / "data"
COINT_CSV = PROCESSED_DIR / "cointegration_results_5pct_noncollinear.csv"

In [14]:
# ── Config ─────────────────────────────────────────────────────────
MAX_PAIRS = 10_000   # None = all ~72k pairs; 10_000 runs in a few minutes
TOP_N = 50

LOOKBACK = 60
ENTRY_Z = 2.0
EXIT_Z = 0.0
CAPITAL = 100_000
COST_BPS = 5
MIN_TRADES = 5

In [15]:
print("Loading cointegration results...")
coint = pd.read_csv(COINT_CSV)
coint = coint.sort_values("pvalue", ascending=True).reset_index(drop=True)
if MAX_PAIRS is not None:
    coint = coint.head(MAX_PAIRS)
pairs_list = list(coint[["ticker1", "ticker2"]].itertuples(index=False, name=None))
all_tickers = list({t for p in pairs_list for t in p})
print(f"Backtesting {len(pairs_list):,} pairs ({len(all_tickers)} unique tickers)...")

Loading cointegration results...
Backtesting 10,000 pairs (1243 unique tickers)...


In [16]:
print("Loading price data...")
prices = load_prices(
    tickers=all_tickers,
    data_dir=DATA_DIR,
    columns=["date", "ticker", "adj_close"],
)
prices["date"] = pd.to_datetime(prices["date"]).dt.date
wide = prices.pivot(index="date", columns="ticker", values="adj_close").sort_index()

max_date = wide.index.max()
end_date = max_date - timedelta(days=365)
wide = wide.loc[wide.index <= end_date]
print(f"Backtest range: {wide.index.min()} to {wide.index.max()} ({len(wide)} days)")

Loading price data...
Backtest range: 2020-01-02 to 2025-02-18 (1289 days)


In [17]:
results = []
for i, (a, b) in enumerate(pairs_list):
    if (i + 1) % 5000 == 0 or i == 0:
        print(f"  {i + 1:,} / {len(pairs_list):,} pairs...")
    res = backtest_pair(wide, a, b, LOOKBACK, ENTRY_Z, EXIT_Z, CAPITAL, COST_BPS)
    if "error" in res:
        continue
    results.append(res)  # keep pnl_series for YoY / geom-mean table

summary = pd.DataFrame([{k: v for k, v in r.items() if k != "pnl_series"} for r in results])

  1 / 10,000 pairs...
  5,000 / 10,000 pairs...
  10,000 / 10,000 pairs...


In [18]:
if summary.empty:
    print("No valid backtest results.")
else:
    summary = summary[summary["n_trades"] >= MIN_TRADES].copy()
    print(f"Pairs with at least {MIN_TRADES} trades: {len(summary):,}")
    summary = summary.sort_values(["sharpe", "n_trades"], ascending=[False, False]).reset_index(drop=True)
    top = summary.head(TOP_N)
    print(f"Top {TOP_N} pairs by Sharpe ratio (then n_trades):")
    display(top)
    pairs_out = [(r["ticker_a"], r["ticker_b"]) for _, r in top.iterrows()]
    print("# PAIRS for pairs_backtest.ipynb:")
    print("PAIRS =", pairs_out)

Pairs with at least 5 trades: 8,971
Top 50 pairs by Sharpe ratio (then n_trades):


,pair,ticker_a,ticker_b,days,total_pnl,total_cost,sharpe,yearly_growth_pct,mean_daily_pnl,std_daily_pnl,n_trades
0,CURB/SOXX,CURB,SOXX,37,35997.97,300.0,7.641,711.83,972.92,2021.31,6
1,CURB/TAN,CURB,TAN,37,31046.02,300.0,6.575,530.60,839.08,2025.80,6
2,CURB/IRDM,CURB,IRDM,37,37981.88,250.0,6.490,795.99,1026.54,2510.90,5
3,PFBC/SOLV,PFBC,SOLV,164,157044.46,450.0,5.708,326.59,957.59,2662.95,9
4,APLS/SOLV,APLS,SOLV,164,182402.82,250.0,4.368,392.95,1112.21,4041.89,5
5,COP/WAY,COP,WAY,113,55658.55,400.0,3.181,168.26,492.55,2458.10,8
6,PMT/SOLV,PMT,SOLV,164,60494.42,350.0,3.098,106.87,368.87,1890.11,7
7,D/GEV,D,GEV,163,57709.65,250.0,2.931,102.25,354.05,1917.74,5
8,CURB/DOW,CURB,DOW,37,9863.28,250.0,2.849,89.78,266.58,1485.30,5
9,APLE/VSTS,APLE,VSTS,288,75195.62,350.0,2.805,63.34,261.10,1477.62,7


# PAIRS for pairs_backtest.ipynb:
PAIRS = [('CURB', 'SOXX'), ('CURB', 'TAN'), ('CURB', 'IRDM'), ('PFBC', 'SOLV'), ('APLS', 'SOLV'), ('COP', 'WAY'), ('PMT', 'SOLV'), ('D', 'GEV'), ('CURB', 'DOW'), ('APLE', 'VSTS'), ('DXC', 'VSTS'), ('CURB', 'INSP'), ('CURB', 'TFX'), ('BNO', 'EMBC'), ('NCLH', 'TIP'), ('EWZ', 'JNJ'), ('EMBC', 'WINA'), ('HAFC', 'SOLV'), ('GDEN', 'VSTS'), ('EMBC', 'KTB'), ('INN', 'PLD'), ('NCLH', 'VXF'), ('IQV', 'OXY'), ('EG', 'MBC'), ('OXY', 'YOU'), ('EMBC', 'KR'), ('EG', 'SOLV'), ('DXC', 'PZZA'), ('NABL', 'SRPT'), ('SPY', 'VTIP'), ('SNCY', 'VIS'), ('EMBC', 'SXI'), ('EMBC', 'GO'), ('EMBC', 'IRDM'), ('NCLH', 'SM'), ('GPI', 'SYK'), ('AHCO', 'SOLV'), ('CCL', 'SNEX'), ('NCLH', 'WAFD'), ('NCLH', 'RJF'), ('BDX', 'TTWO'), ('GPN', 'VSTS'), ('REYN', 'VAW'), ('DGX', 'WMT'), ('CERT', 'TIP'), ('NBHC', 'WAFD'), ('HWM', 'NRG'), ('AXL', 'UPS'), ('CCL', 'EBAY'), ('REYN', 'VFH')]


In [19]:
import numpy as np
top_pairs = set(zip(top["ticker_a"], top["ticker_b"]))
top_results = [r for r in results if (r["ticker_a"], r["ticker_b"]) in top_pairs]
yoy_list = []
geom_mean_list = []
for r in top_results:
    pair_name = r["pair"]
    pnl = r.get("pnl_series")
    if pnl is None or pnl.empty:
        continue
    pnl = pnl.copy()
    if not hasattr(pnl.index, "year"):
        pnl.index = pd.to_datetime(pnl.index)
    by_year = pnl.groupby(pnl.index.year).sum()
    yearly_pnl = by_year.iloc[:, 0] if by_year.ndim > 1 else by_year
    yearly_ret_pct = (yearly_pnl / CAPITAL * 100).round(2)
    yoy_list.append(pd.DataFrame({pair_name: yearly_ret_pct}))
    r_frac = yearly_pnl / CAPITAL
    g = (np.prod(1 + r_frac) ** (1 / len(r_frac)) - 1) * 100 if len(r_frac) > 0 else np.nan
    geom_mean_list.append({"pair": pair_name, "geom_mean_ret_pct": round(g, 2)})
yoy_table = pd.concat(yoy_list, axis=1) if yoy_list else pd.DataFrame()
yoy_table.index.name = "year"
print("Year-over-year return (% of CAPITAL per year):")
display(yoy_table)
geom_mean_df = pd.DataFrame(geom_mean_list)
print("Geometric mean of yearly returns (%):")
display(geom_mean_df)

Year-over-year return (% of CAPITAL per year):


/var/folders/xz/2ks9m60d53gg2lrfyqwfk7kr0000gn/T/ipykernel_14648/2240866599.py:19: RuntimeWarning: invalid value encountered in scalar power
  g = (np.prod(1 + r_frac) ** (1 / len(r_frac)) - 1) * 100 if len(r_frac) > 0 else np.nan
/var/folders/xz/2ks9m60d53gg2lrfyqwfk7kr0000gn/T/ipykernel_14648/2240866599.py:19: RuntimeWarning: invalid value encountered in scalar power
  g = (np.prod(1 + r_frac) ** (1 / len(r_frac)) - 1) * 100 if len(r_frac) > 0 else np.nan


,NCLH/VXF,CCL/EBAY,CURB/TAN,CURB/SOXX,NCLH/TIP,OXY/YOU,DXC/VSTS,NCLH/WAFD,NCLH/RJF,EMBC/IRDM,...,HWM/NRG,EWZ/JNJ,SPY/VTIP,DXC/PZZA,COP/WAY,NBHC/WAFD,D/GEV,INN/PLD,NABL/SRPT,EMBC/KTB
year,,,,,,,,,,,,,,,,,,,,,
2020,189.21,392.97,NaN,NaN,786.57,NaN,NaN,29.02,202.66,NaN,...,38.10,274.15,91.67,190.90,NaN,77.88,NaN,361.52,NaN,NaN
2021,280.86,248.17,NaN,NaN,174.73,NaN,NaN,554.90,560.27,NaN,...,107.77,134.73,59.39,298.11,NaN,109.83,NaN,59.63,53.96,NaN
2022,158.77,101.00,NaN,NaN,-0.69,99.12,NaN,231.28,280.80,299.05,...,72.72,288.61,-22.81,243.95,NaN,362.12,NaN,114.34,126.76,261.10
2023,255.85,158.12,NaN,NaN,-60.38,17.96,16.02,49.74,171.25,302.05,...,-0.46,287.95,167.54,40.89,NaN,213.04,NaN,151.41,75.33,149.12
2024,130.26,45.39,7.41,8.32,191.32,50.92,162.67,411.91,209.93,132.43,...,500.35,82.38,145.18,59.43,55.66,335.90,49.51,-24.97,125.41,126.77
2025,0.00,68.20,23.64,27.68,0.00,-3.23,24.78,0.00,29.30,0.00,...,66.60,-4.39,129.13,3.60,0.00,0.00,8.20,41.33,36.08,247.85


Geometric mean of yearly returns (%):


,pair,geom_mean_ret_pct
0,NCLH/VXF,148.16
1,CCL/EBAY,145.28
2,CURB/TAN,15.24
3,CURB/SOXX,17.60
4,NCLH/TIP,74.18
5,OXY/YOU,36.09
6,DXC/VSTS,56.09
7,NCLH/WAFD,144.68
8,NCLH/RJF,206.39
9,EMBC/IRDM,147.12


In [20]:
import numpy as np

# Full result dicts for top N (with pnl_series)
top_pairs = set(zip(top["ticker_a"], top["ticker_b"]))
top_results = [r for r in results if (r["ticker_a"], r["ticker_b"]) in top_pairs]

yoy_list = []
geom_mean_list = []
for r in top_results:
    pair_name = r["pair"]
    pnl = r.get("pnl_series")
    if pnl is None or pnl.empty:
        continue
    pnl = pnl.copy()
    if not hasattr(pnl.index, "year"):
        pnl.index = pd.to_datetime(pnl.index)
    by_year = pnl.groupby(pnl.index.year).sum()
    yearly_pnl = by_year.iloc[:, 0] if by_year.ndim > 1 else by_year
    yearly_ret_pct = (yearly_pnl / CAPITAL * 100).round(2)
    yoy_list.append(pd.DataFrame({pair_name: yearly_ret_pct}))
    r_frac = yearly_pnl / CAPITAL
    g = (np.prod(1 + r_frac) ** (1 / len(r_frac)) - 1) * 100 if len(r_frac) > 0 else np.nan
    geom_mean_list.append({"pair": pair_name, "geom_mean_ret_pct": round(g, 2)})

yoy_table = pd.concat(yoy_list, axis=1) if yoy_list else pd.DataFrame()
yoy_table.index.name = "year"
print("Year-over-year return (% of CAPITAL per year):")
display(yoy_table)

geom_mean_df = pd.DataFrame(geom_mean_list)
print("Geometric mean of yearly returns (%):")
display(geom_mean_df)

Year-over-year return (% of CAPITAL per year):


/var/folders/xz/2ks9m60d53gg2lrfyqwfk7kr0000gn/T/ipykernel_14648/3237960600.py:22: RuntimeWarning: invalid value encountered in scalar power
  g = (np.prod(1 + r_frac) ** (1 / len(r_frac)) - 1) * 100 if len(r_frac) > 0 else np.nan
/var/folders/xz/2ks9m60d53gg2lrfyqwfk7kr0000gn/T/ipykernel_14648/3237960600.py:22: RuntimeWarning: invalid value encountered in scalar power
  g = (np.prod(1 + r_frac) ** (1 / len(r_frac)) - 1) * 100 if len(r_frac) > 0 else np.nan


,NCLH/VXF,CCL/EBAY,CURB/TAN,CURB/SOXX,NCLH/TIP,OXY/YOU,DXC/VSTS,NCLH/WAFD,NCLH/RJF,EMBC/IRDM,...,HWM/NRG,EWZ/JNJ,SPY/VTIP,DXC/PZZA,COP/WAY,NBHC/WAFD,D/GEV,INN/PLD,NABL/SRPT,EMBC/KTB
year,,,,,,,,,,,,,,,,,,,,,
2020,189.21,392.97,NaN,NaN,786.57,NaN,NaN,29.02,202.66,NaN,...,38.10,274.15,91.67,190.90,NaN,77.88,NaN,361.52,NaN,NaN
2021,280.86,248.17,NaN,NaN,174.73,NaN,NaN,554.90,560.27,NaN,...,107.77,134.73,59.39,298.11,NaN,109.83,NaN,59.63,53.96,NaN
2022,158.77,101.00,NaN,NaN,-0.69,99.12,NaN,231.28,280.80,299.05,...,72.72,288.61,-22.81,243.95,NaN,362.12,NaN,114.34,126.76,261.10
2023,255.85,158.12,NaN,NaN,-60.38,17.96,16.02,49.74,171.25,302.05,...,-0.46,287.95,167.54,40.89,NaN,213.04,NaN,151.41,75.33,149.12
2024,130.26,45.39,7.41,8.32,191.32,50.92,162.67,411.91,209.93,132.43,...,500.35,82.38,145.18,59.43,55.66,335.90,49.51,-24.97,125.41,126.77
2025,0.00,68.20,23.64,27.68,0.00,-3.23,24.78,0.00,29.30,0.00,...,66.60,-4.39,129.13,3.60,0.00,0.00,8.20,41.33,36.08,247.85


Geometric mean of yearly returns (%):


,pair,geom_mean_ret_pct
0,NCLH/VXF,148.16
1,CCL/EBAY,145.28
2,CURB/TAN,15.24
3,CURB/SOXX,17.60
4,NCLH/TIP,74.18
5,OXY/YOU,36.09
6,DXC/VSTS,56.09
7,NCLH/WAFD,144.68
8,NCLH/RJF,206.39
9,EMBC/IRDM,147.12
